# 3DGS: full handwritten Metal render pipeline

This experimental notebook keeps PyTorch as the tensor owner and dispatch layer,
but moves the render front-end and rasterization to handwritten Metal kernels.

Pipeline:

```text
Gaussian setup / projection + spherical harmonics
→ rectangular screen-space AABB
→ hierarchical exclusive scan
→ Gaussian/tile intersection emission
→ stable 4-bit LSD radix sort by (tile_id, depth)
→ dense tile counts + exclusive scan
→ one Metal threadgroup per tile
→ front-to-back alpha compositing
```

The Python renderer is now the `metal_renderer/` package; Metal source stays in
`metal_kernels/`. `GaussianData` adapts the flat checkpoint SH layout to the
renderer-facing coefficient layout, while `MetalRenderer.render(c2w)` only needs
the per-frame camera transform.

The selected SH level count is a compile-time define in `gaussian_setup.metal`.
SH color is evaluated once per visible Gaussian in the setup kernel rather than in
the tile rasterizer, because the view direction is constant for a Gaussian during
one camera render. Recomputing it per pixel/tile would only duplicate work.

There is no explicit `torch.mps.synchronize()` inside `render()`. The benchmark
synchronizes outside the renderer. The one unavoidable GPU→CPU synchronization is
still the scalar `K = number of Gaussian/tile intersections`, needed before PyTorch
can allocate the variable-length intersection buffers.


In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch

from metal_renderer import GaussianData, MetalRenderer
from pytorch_reference_renderer import gaussian_rasterization_pytorch
from util import load_cameras, scale_intrinsics

In [ ]:
scene = "bonsai"
device = torch.device("mps")
SH_LEVELS = 4

if not torch.backends.mps.is_available():
    raise RuntimeError("This notebook requires an Apple Silicon MPS device")

gaussian_data = GaussianData.from_checkpoint(
    "out_bonsai",
    device=device,
    sh_levels=SH_LEVELS,
)

cam_parameters = np.load(
    f"out_colmap/{scene}/cam_meta.npy",
    allow_pickle=True,
).item()

H_source = cam_parameters["height"]
W_source = cam_parameters["width"]
fx_source = cam_parameters["fx"]
fy_source = cam_parameters["fy"]
cx_source = W_source / 2
cy_source = H_source / 2

H = H_source // 2
W = W_source // 2
fx, fy, cx, cy = scale_intrinsics(
    H,
    W,
    H_source,
    W_source,
    fx_source,
    fy_source,
    cx_source,
    cy_source,
)

c2ws, image_paths = load_cameras(
    f"out_colmap/{scene}/cameras.npy",
    f"image_data/{scene}/images_2",
)

CAM_ID = 10
c2w = c2ws[CAM_ID].to(device)
image_path = image_paths[CAM_ID]

renderer = MetalRenderer(
    gaussian_data,
    H=H,
    W=W,
    fx=fx,
    fy=fy,
    cx=cx,
    cy=cy,
)

print(f"Gaussians: {gaussian_data.num_gaussians:,}")
print(f"SH levels: {gaussian_data.sh_levels}")
print(f"SH coefficients/color: {gaussian_data.sh_coefficient_count}")
print(f"Image: {W} × {H}")
print(f"Camera: {CAM_ID}")

## Metal implementation notes

`gaussian_setup.metal` performs SH color evaluation, projection, frustum filtering,
covariance projection/stabilization, inverse conic construction and rectangular 3σ
AABB construction. Invisible Gaussians simply emit `tile_count = 0`, so no PyTorch
mask compaction is needed.

`scan.metal` provides a hierarchical exclusive scan. `binning.metal` emits the
variable-length Gaussian/tile records. `radix_sort.metal` uses stable 4-bit LSD
passes: depth bits first, then tile-id bits, yielding `(tile_id, depth)` order.

Finally `tile_rasterizer.metal` consumes dense tile offsets and performs
front-to-back alpha compositing with one threadgroup per image tile.


In [ ]:
def synchronize():
    torch.mps.synchronize()


def measure(renderer, *args):
    synchronize()
    started_at = perf_counter()
    image = renderer(*args)
    synchronize()
    return image, perf_counter() - started_at


def render_pytorch(c2w):
    color = gaussian_data.evaluate_color(c2w)
    return gaussian_rasterization_pytorch(
        gaussian_data.positions,
        color,
        gaussian_data.opacity_raw,
        gaussian_data.sigma,
        c2w,
        H,
        W,
        fx,
        fy,
        cx,
        cy,
    )


# Compile Metal libraries and warm allocator/caches before timing.
_ = renderer.render(c2w)
synchronize()

img_pytorch, pytorch_seconds = measure(render_pytorch, c2w)
img_metal, metal_seconds = measure(renderer.render, c2w)

print(f"PyTorch reference: {pytorch_seconds * 1_000:.2f} ms")
print(f"Full Metal pipeline: {metal_seconds * 1_000:.2f} ms")
print(f"Speedup: {pytorch_seconds / metal_seconds:.2f}×")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(img_pytorch.detach().cpu().numpy())
axes[0].set_title("PyTorch reference")
axes[0].axis("off")

axes[1].imshow(img_metal.detach().cpu().numpy())
axes[1].set_title("Full Metal pipeline")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
stats = renderer.last_stats

print("Metal pipeline diagnostics")
for key, value in stats.items():
    print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

In [ ]:
absolute_difference = (img_pytorch - img_metal).abs()

strict_atol = 2e-3
sparse_outlier_atol = 1e-2
maximum_sparse_outlier_fraction = 1e-5

max_absolute_difference = absolute_difference.max().item()
mean_absolute_difference = absolute_difference.mean().item()
strict_mismatches = (absolute_difference > strict_atol).sum().item()
strict_mismatch_fraction = strict_mismatches / absolute_difference.numel()

print(f"Maximum absolute difference: {max_absolute_difference:.8f}")
print(f"Mean absolute difference: {mean_absolute_difference:.8f}")
print(f"Values with absolute difference > {strict_atol}: {strict_mismatches}")
print(f"Sparse mismatch fraction: {strict_mismatch_fraction:.10%}")

assert strict_mismatch_fraction <= maximum_sparse_outlier_fraction, (
    f"Too many values differ by more than {strict_atol}: "
    f"{strict_mismatches} / {absolute_difference.numel()}"
)

torch.testing.assert_close(
    img_metal,
    img_pytorch,
    rtol=1e-4,
    atol=sparse_outlier_atol,
)
print("Pixel-wise comparison passed with sparse float32 outliers allowed.")

plt.figure(figsize=(8, 6))
plt.imshow(absolute_difference.max(dim=-1).values.detach().cpu().numpy())
plt.title("Maximum absolute difference per pixel")
plt.colorbar()
plt.axis("off")
plt.show()
